# 02 — Feature Engineering

**Objective:** Transform raw data into a modeling-ready feature set.

**Input:** `data/raw/churn-bigml-80_raw.csv`  
**Output:** `data/processed/churn_train.csv`, `data/processed/churn_test.csv`, `models/scaler.pkl`

## 1. Imports

In [ ]:
import sys
import os
sys.path.append("../")  # allow 'from src.x import y' from notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

from src.preprocessing import run_preprocessing_pipeline

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

## 2. Load Cleaned Data

In [ ]:
clean_path = Path("../data/processed/churn_clean.csv")
if not clean_path.exists():
    print("churn_clean.csv not found — running preprocessing pipeline...")
    run_preprocessing_pipeline(
        input_path="../data/raw/churn-bigml-80_raw.csv",
        output_path="../data/processed/churn_clean.csv",
    )

df = pd.read_csv(clean_path)
print(f"Loaded: {df.shape}")
df.head()

## 3. Drop Multicollinear Charge Columns

In [ ]:
# Charge columns are near-perfect linear functions of minutes columns
# (charge = minutes x fixed rate), causing multicollinearity.
# Keep minutes; drop charges.
charge_cols = [
    "Total day charge", "Total eve charge",
    "Total night charge", "Total intl charge",
]
df = df.drop(columns=[c for c in charge_cols if c in df.columns])
print(f"Shape after dropping charge columns: {df.shape}")

## 4. Feature Engineering

In [ ]:
# --- Aggregation features ---

# Total minutes across ALL periods (including intl)
minute_cols = ["Total day minutes", "Total eve minutes",
               "Total night minutes", "Total intl minutes"]
df["total_minutes"] = df[minute_cols].sum(axis=1)

# Total calls across all periods
call_cols_all = ["Total day calls", "Total eve calls",
                 "Total night calls", "Total intl calls"]
df["total_calls"] = df[call_cols_all].sum(axis=1)

# Calls per minute — usage intensity proxy
df["calls_per_minute"] = np.where(
    df["total_minutes"] > 0,
    df["total_calls"] / df["total_minutes"],
    0.0,
)

# --- Business-derived risk flags ---

# >=4 service calls is a well-documented churn signal
df["high_service_calls"] = (df["Customer service calls"] >= 4).astype(int)

# Has intl plan but 0 intl usage — paying for unused service
if "International plan" in df.columns and "Total intl minutes" in df.columns:
    df["intl_plan_no_usage"] = (
        (df["International plan"] == 1) & (df["Total intl minutes"] == 0)
    ).astype(int)

    # No intl plan but heavy intl usage — unexpected overage bills
    df["intl_high_usage_no_plan"] = (
        (df["International plan"] == 0) & (df["Total intl minutes"] > 10)
    ).astype(int)

# Has voicemail plan but 0 messages — plan dissatisfaction
if "Voice mail plan" in df.columns and "Number vmail messages" in df.columns:
    df["vmail_mismatch"] = (
        (df["Voice mail plan"] == 1) & (df["Number vmail messages"] == 0)
    ).astype(int)

new_features = [
    "total_minutes", "total_calls", "calls_per_minute",
    "high_service_calls", "intl_plan_no_usage",
    "intl_high_usage_no_plan", "vmail_mismatch",
]
print(f"New features: {new_features}")
print(f"Shape after feature engineering: {df.shape}")

**Executive interpretation:** Risk flags encode domain knowledge directly — customers with 4+ service calls churn at ~50%, and unexpected international bills are a known churn trigger.

## 5. Correlation of New Features with Churn

In [ ]:
available = [f for f in new_features if f in df.columns]
corr_with_churn = (
    df[available + ["Churn"]].corr()["Churn"]
    .drop("Churn")
    .sort_values(ascending=False)
)

plt.figure(figsize=(7, 4))
corr_with_churn.plot(
    kind="barh",
    color=["tomato" if v > 0 else "steelblue" for v in corr_with_churn]
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation of Engineered Features with Churn")
plt.xlabel("Pearson Correlation")
plt.tight_layout()
plt.show()
print(corr_with_churn)

## 6. Train / Test Split

In [ ]:
target = "Churn"
X = df.drop(columns=[target])
y = df[target]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X = X[numeric_cols]

# Split BEFORE scaling — prevents leakage of test statistics into scaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

## 7. StandardScaler (fit on train only)

In [ ]:
scaler = StandardScaler()

# fit_transform on train, transform-only on test
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=numeric_cols
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=numeric_cols
)

joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler saved to models/scaler.pkl")
X_train_scaled.describe().loc[["mean", "std"]].round(3)

## 8. SMOTE — Handle Class Imbalance (train only)

In [ ]:
print("Before SMOTE:")
print(pd.Series(y_train).value_counts())
print(f"Churn rate: {pd.Series(y_train).mean():.1%}")

# SMOTE comes AFTER scaling:
# fit_resample returns a numpy array, so we wrap back into DataFrame
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_scaled, y_train = sm.fit_resample(X_train_scaled, y_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=numeric_cols)
    print(f"\nAfter SMOTE — train distribution:")
    print(pd.Series(y_train).value_counts())
    print(f"New train shape: {X_train_scaled.shape}")
except ImportError:
    print("imbalanced-learn not installed — skipping SMOTE.")
    print("Install with: pip install imbalanced-learn")

**Executive interpretation:** SMOTE synthesises new minority-class (churn) samples to create a balanced 50/50 training distribution. The test set is **not** resampled — it retains the real ~14% churn rate for honest evaluation.

## 9. Save Train / Test Sets

In [ ]:
train_out = X_train_scaled.copy()
train_out[target] = y_train.values if hasattr(y_train, "values") else y_train
train_out.to_csv("../data/processed/churn_train.csv", index=False)

test_out = X_test_scaled.copy()
test_out[target] = y_test.values
test_out.to_csv("../data/processed/churn_test.csv", index=False)

print(f"Train saved: data/processed/churn_train.csv  shape={train_out.shape}")
print(f"Test saved : data/processed/churn_test.csv   shape={test_out.shape}")

## 10. Summary

| Step | Action | Rationale |
|------|--------|-----------|
| Drop charge cols | Removed 4 charge columns | Near-perfect correlation with minutes → multicollinearity |
| Aggregation | `total_minutes`, `total_calls`, `calls_per_minute` | Holistic usage view across all periods |
| Risk flags | `high_service_calls`, `intl_high_usage_no_plan`, `intl_plan_no_usage`, `vmail_mismatch` | Domain knowledge churn signals |
| Split first | 80/20 stratified split before scaling | Prevents data leakage |
| StandardScaler | fit on train only | Normalises features for LR and Neural Network |
| SMOTE | Train set only, after scaling | Balances ~14% minority class; test set untouched for honest eval |